In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
train_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/mohs_hardness/train.csv'
train_df = pd.read_csv(train_path)

# Display the first few rows of the dataframe
train_df.head()

# Check the information of the dataset
train_df.info()

# Check for missing values
train_df.isnull().sum()

# Descriptive statistics
train_df.describe()

# Distinguish column types
numeric_features = train_df.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = train_df.select_dtypes(exclude=[np.number]).columns.tolist()

# Visualize the distribution of numeric features
for feature in numeric_features:
    plt.figure(figsize=(10, 4))
    sns.histplot(train_df[feature], kde=True)
    plt.title(f'Distribution of {feature}')
    plt.show()

# Visualize the distribution of categorical features
for feature in categorical_features:
    plt.figure(figsize=(10, 4))
    sns.countplot(data=train_df, x=feature)
    plt.title(f'Distribution of {feature}')
    plt.xticks(rotation=45)
    plt.show()

# Correlation matrix for numeric features
plt.figure(figsize=(12, 8))
corr_matrix = train_df[numeric_features].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix of Numeric Features')
plt.show()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8325 entries, 0 to 8324
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     8325 non-null   int64  
 1   allelectrons_Total     8325 non-null   float64
 2   density_Total          8325 non-null   float64
 3   allelectrons_Average   8325 non-null   float64
 4   val_e_Average          8325 non-null   float64
 5   atomicweight_Average   8325 non-null   float64
 6   ionenergy_Average      8325 non-null   float64
 7   el_neg_chi_Average     8325 non-null   float64
 8   R_vdw_element_Average  8325 non-null   float64
 9   R_cov_element_Average  8325 non-null   float64
 10  zaratio_Average        8325 non-null   float64
 11  density_Average        8325 non-null   float64
 12  Hardness               8325 non-null   float64
dtypes: float64(12), int64(1)
memory usage: 845.6 KB


In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df)
print("column_info")
print(column_info)


2025-08-30 23:05:18.996 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': [], 'Numeric': ['id', 'allelectrons_Total', 'density_Total', 'allelectrons_Average', 'val_e_Average', 'atomicweight_Average', 'ionenergy_Average', 'el_neg_chi_Average', 'R_vdw_element_Average', 'R_cov_element_Average', 'zaratio_Average', 'density_Average', 'Hardness'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, StandardScale

# Copy the DataFrame to avoid modifying the original data
train_df_copy = train_df.copy()

# Handling missing values
# Since there are no missing values in the dataset, this step is not necessary.
# However, if there were, we would use the FillMissingValue tool to handle them.
# For example, to fill missing values in numeric features with the mean:
# fill_missing = FillMissingValue(numeric_features, strategy='mean')
# train_df_copy = fill_missing.fit_transform(train_df_copy)

# Scaling numerical features
# We will scale all numeric features except the label 'Hardness'
numeric_features_to_scale = [col for col in numeric_features if col != 'Hardness']
scaler = StandardScale(numeric_features_to_scale)
train_df_copy = scaler.fit_transform(train_df_copy)

# Display the first few rows of the preprocessed DataFrame
train_df_copy.head()


,id,allelectrons_Total,density_Total,allelectrons_Average,val_e_Average,atomicweight_Average,ionenergy_Average,el_neg_chi_Average,R_vdw_element_Average,R_cov_element_Average,zaratio_Average,density_Average,Hardness
0,-1.028644,-0.449761,-0.774773,-0.672137,1.136606,-0.643432,2.297123,1.440287,-0.921911,-0.435357,-0.204818,-0.836903,6.0
1,-1.604442,-0.293378,-0.068134,0.566681,-0.795740,0.693478,-1.370169,-1.157492,0.920790,1.552238,-0.301943,1.334983,3.3
2,-0.703468,-0.141595,-0.062591,-0.469638,-0.252268,-0.481614,0.070652,-0.190790,-0.065247,0.042074,-0.105408,-0.174700,5.3
3,-1.157450,0.106778,0.514155,3.171377,-0.312654,3.199840,-1.713351,-1.685010,1.805427,2.345793,-1.221058,2.893320,1.8
4,-1.548859,0.014788,0.585200,0.026683,0.170433,0.051579,-0.340901,-0.094990,0.109582,0.157955,-0.311987,-0.109614,5.5


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df_copy)
print("column_info")
print(column_info)


column_info
{'Category': [], 'Numeric': ['id', 'allelectrons_Total', 'density_Total', 'allelectrons_Average', 'val_e_Average', 'atomicweight_Average', 'ionenergy_Average', 'el_neg_chi_Average', 'R_vdw_element_Average', 'R_cov_element_Average', 'zaratio_Average', 'density_Average', 'Hardness'], 'Datetime': [], 'Others': []}


In [5]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, median_absolute_error
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from metagpt.tools.libs.data_preprocess import FillMissingValue, StandardScale

# Split the data into features and target
X = train_df_copy.drop(columns=['Hardness', 'id'])
y = train_df_copy['Hardness']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the model
xgb = XGBRegressor(random_state=42)

# Define the parameter grid for GridSearchCV
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.1, 0.2]
}

# Perform grid search to find the best hyperparameters
grid_search = GridSearchCV(estimator=xgb, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Get the best model
best_model = grid_search.best_estimator_

# Predict on the test set
y_pred = best_model.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
medae = median_absolute_error(y_test, y_pred)

print(f'Mean Squared Error: {mse}')
print(f'Mean Absolute Error: {mae}')
print(f'Median Absolute Error: {medae}')


Mean Squared Error: 1.5063159339392647
Mean Absolute Error: 0.9005149202518634
Median Absolute Error: 0.6246814727783203
